# Diabetes Risk Prediction: Data Science Lifecycle

**Author:** Adegboyega Samuel

This project uses routine health measurements to study diabetes risk. It follows the full data science process: defining the problem, preparing the data, exploring patterns, building models, and explaining the results.

The model is meant for analysis and demonstration only. It should not be used to diagnose diabetes.

## 1. The Question

**Can simple health measurements help predict whether someone is likely to have diabetes?**

The measurements include glucose, BMI, age, blood pressure, insulin, and family history score. The goal is not to replace doctors. The goal is to see whether these measurements can help identify people who may need proper follow-up testing.

## 2. The Data

The dataset has **768 patient records**. Each row is one patient. Each patient has 8 health measurements and one final outcome:

- `0` means the patient did not test positive for diabetes.
- `1` means the patient tested positive for diabetes.

The data comes from the Pima Indians Diabetes dataset. A key limitation is that the dataset only represents one specific population, so the result should not be assumed to work for everyone.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from data_prep import ZERO_AS_MISSING, load_raw, load_clean

raw = load_raw(PROJECT_ROOT / 'data' / 'pima_diabetes_raw.csv')
clean = load_clean(PROJECT_ROOT / 'data' / 'pima_diabetes_raw.csv')

raw.shape

## 3. Cleaning the Data

Some columns used `0` to mean a measurement was missing. That is a problem because some zero values are medically impossible. For example, a living patient cannot have blood pressure of zero.

So the project treats those zeros as missing values instead of real measurements.

In [ ]:
missing_zero_counts = (raw[ZERO_AS_MISSING] == 0).sum().sort_values(ascending=False)
missing_zero_percent = (missing_zero_counts / len(raw) * 100).round(1)

missing_summary = missing_zero_counts.to_frame('rows_with_zero_placeholder')
missing_summary['percent_of_dataset'] = missing_zero_percent
missing_summary

### Missing Values Before Cleaning

This chart shows how many impossible zero values appeared in each column before cleaning.

![Missing values before cleaning](../results/figures/missing_values_before_cleaning.png)

**What this shows:** insulin and skin thickness had the biggest missing-data problem. This matters because a model can learn the wrong pattern if missing values are treated as real zeros.

### Before and After Cleaning

This chart compares the messy data before cleaning with the cleaned data after impossible zeros were fixed.

![Before and after cleaning](../results/figures/before_after_cleaning.png)

**What this shows:** the tall bars at zero disappear after cleaning. That means the project no longer treats missing tests as real medical measurements.

## 4. Understanding the Data

Before building a model, we first check whether the measurements actually show useful patterns. This helps us avoid blindly trusting the model.

In [ ]:
outcome_balance = clean['diabetes'].value_counts().sort_index().rename(index={0: 'No diabetes', 1: 'Diabetes'})
outcome_rate = (outcome_balance / len(clean) * 100).round(1)
outcome_balance.to_frame('patients').assign(percent=outcome_rate)

### Comparing Patients With and Without Diabetes

These boxplots compare each measurement for patients without diabetes and patients with diabetes.

![Boxplots by outcome](../results/figures/boxplots_by_outcome.png)

**What this shows:** glucose, BMI, and age show clearer differences between the two groups. Blood pressure and skin thickness are less clearly separated.

### Diabetes Rate From Low to High Values

Each measurement is split into four groups from low to high. The bars show the diabetes rate in each group.

![Diabetes rate by bins](../results/figures/rate_by_bins.png)

**What this shows:** glucose has the strongest pattern. Patients in the highest glucose group have a much higher diabetes rate than patients in the lowest glucose group.

### Correlation Between Measurements

A correlation chart shows which measurements move together. A higher number means a stronger relationship.

![Correlation heatmap](../results/figures/correlation_heatmap.png)

**What this shows:** glucose has the strongest single relationship with diabetes. This makes sense because glucose is directly connected to how diabetes is diagnosed.

## 5. Building the Models

Two models were tested:

1. **Logistic Regression:** a simpler model that is easier to explain.
2. **Random Forest:** a more flexible model that can find more complex patterns.

The data was split into training data and test data. The models learned from the training data, then were judged on the test data they had not seen before.

In [ ]:
import pandas as pd
from sklearn.metrics import precision_score, recall_score, roc_auc_score
from model import build_logistic_pipeline, build_rf_pipeline, split_data

X_train, X_test, y_train, y_test = split_data(clean)
models = {
    'Logistic Regression': build_logistic_pipeline().fit(X_train, y_train),
    'Random Forest': build_rf_pipeline().fit(X_train, y_train),
}

rows = []
for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    rows.append({
        'model': name,
        'auc': round(roc_auc_score(y_test, proba), 3),
        'recall': round(recall_score(y_test, pred), 3),
        'precision': round(precision_score(y_test, pred), 3),
    })

pd.DataFrame(rows)

## 6. Model Results

The models are not perfect, but they perform better than guessing. The most important thing is to understand the type of mistakes they make.

### ROC Curve

The ROC curve checks whether the model can rank higher-risk patients above lower-risk patients.

![ROC curve](../results/figures/roc_curve.png)

**What this shows:** both model curves sit above the gray random-guessing line. That means both models learned a real pattern from the data.

### Confusion Matrices

A confusion matrix shows exactly how many test patients were correctly or incorrectly classified.

![Confusion matrices](../results/figures/confusion_matrices.png)

**What this shows:** the models correctly identify many non-diabetes cases, but they still miss some real diabetes cases. This is why the model should not be used as a diagnosis tool.

### Threshold Tradeoff

A model gives a risk score. We choose a cutoff to decide when to label someone as high risk. Lowering the cutoff catches more true cases, but it also creates more false alarms.

![Threshold tradeoff](../results/figures/threshold_tradeoff.png)

**What this shows:** if the goal is screening, a lower threshold may be useful because it catches more diabetes cases. But it also means more people without diabetes may be flagged for follow-up.

## 7. What the Models Learned

Feature importance tells us which measurements the models relied on most.

### Feature Importance

![Feature importance](../results/figures/feature_importance.png)

**What this shows:** glucose is the most important measurement by far. BMI and age also matter. This matches what we saw earlier in the charts, which makes the result more believable.

## 8. Final Takeaways

- Glucose is the strongest signal in the dataset.
- BMI and age also help predict diabetes risk.
- The models perform better than random guessing.
- The models still miss some real diabetes cases.
- This project is useful as a data science demonstration, but it should not be used for real medical decisions without much more testing.

A responsible real-world version would need more diverse data, clinical review, careful threshold selection, and repeated testing over time.